# Tariff example usage
This notebook demonstrates how to use the `Tariff` class to calculate energy costs based on a given tariff structure. We will create a sample tariff and then use it to compute the cost of energy consumption over a specified period.

### 1. Creating an index
Most tariffs are based on one or more indexes. For example, a common structure is to have a fixed cost component and a variable cost component that depends on the day-ahead electricity price. Let's create an index for the day-ahead price using the `EntsoeDayAheadIndex`:

In [3]:
from os import environ

from energy_cost.index import EntsoeDayAheadIndex, Index

Index.register("Belpex15min", EntsoeDayAheadIndex(country_code="BE", api_key=environ["ENTSOE_API_KEY"]))

### 2. Defining a tariff
The easiest way to define a tariff is to specify it in a YAML file. This allows for a clear separation of the tariff definition from the code that uses it. You can find some examples in `data/tariffs/`. 

The yaml structure is as follows:

```yaml
supplier: "Foo" # The energy supplier providing the tariff
product: "Bar" # The specific product or plan offered by the supplier
by_meter_type: 
  single_rate: # The metering type, eg. single_rate, tou_peak, tou_offpeak, ...
  # The ordered list of price formulas that apply to this metering type. The formulas are assumed to be valid from their start date until the start date of the next formula, or indefinitely if there is no next formula.
  - start: 2025-01-01T00:00:00+01 # The start date of the formula, in ISO 8601 format
    formula:
      constant_cost: 16.25 # The fixed cost component in €/MWh
      variable_costs: # The variable cost component depends on one or more indexes, which are multiplied by a scalar and summed to calculate the cost
      - index: Belpex15min # The index used for the variable cost, in €/MWh
        scalar: 1.08 # The scalar applied to the index to calculate the variable cost
      - index: BelpexRLPO # Another index used in the formula, in €/MWh
        scalar: 0.95 
```

In [4]:
from energy_cost.tariff import Tariff

tariff = Tariff.from_yaml("../data/tariffs/foo/dynamic_tariff.yml")
tariff

Tariff(supplier='foo', product='dynamic tariff', defaults={<PowerDirection.INJECTION: 'injection'>: {<CostType.ENERGY: 'energy'>: [TimedPriceFormula(start=datetime.datetime(2025, 1, 1, 0, 0, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600))), formula=PriceFormula(constant_cost=-14.56, variable_costs=[IndexAdder(index='Belpex15min', scalar=0.9876)]))]}, <PowerDirection.CONSUMPTION: 'consumption'>: {<CostType.RENEWABLE_CERTIFICATES: 'renewable_certificates'>: [TimedPriceFormula(start=datetime.datetime(2025, 1, 1, 0, 0, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600))), formula=PriceFormula(constant_cost=12.0, variable_costs=[]))], <CostType.CHP_CERTIFICATES: 'chp_certificates'>: [TimedPriceFormula(start=datetime.datetime(2025, 1, 1, 0, 0, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600))), formula=PriceFormula(constant_cost=4.56, variable_costs=[]))]}}, by_meter_type={<MeterType.SINGLE_RATE: 'single_rate'>: {<PowerDirection.CONSUMPTION: 'consumption'>: {<CostT

### 3. Calculating cost per energy consumption in a given period
Once we have defined our tariff, we can use it to calculate the cost of energy consumption over a specified period. This is done by calling the `get_cost` method of the `Tariff` class, which returns a DataFrame with the cost for each time step in the given period.

In [5]:
import datetime as dt

from energy_cost.tariff import MeterType, PowerDirection

start = dt.datetime.fromisoformat("2026-03-08 00:00:00+01:00")
end = dt.datetime.fromisoformat("2026-03-10 00:00:00+01:00")
resolution = dt.timedelta(minutes=15)
tariff.get_cost(MeterType.SINGLE_RATE, PowerDirection.CONSUMPTION, start, end, resolution)

,timestamp,renewable_certificates,chp_certificates,energy,total
0,2026-03-08 00:00:00+01:00,12.0,4.56,182.7256,199.2856
1,2026-03-08 00:15:00+01:00,12.0,4.56,172.7503,189.3103
2,2026-03-08 00:30:00+01:00,12.0,4.56,167.5720,184.1320
3,2026-03-08 00:45:00+01:00,12.0,4.56,161.8894,178.4494
4,2026-03-08 01:00:00+01:00,12.0,4.56,172.3936,188.9536
...,...,...,...,...,...
187,2026-03-09 22:45:00+01:00,12.0,4.56,175.7392,192.2992
188,2026-03-09 23:00:00+01:00,12.0,4.56,187.1782,203.7382
189,2026-03-09 23:15:00+01:00,12.0,4.56,180.7945,197.3545
190,2026-03-09 23:30:00+01:00,12.0,4.56,173.8696,190.4296
